# Module 3 — Class 4: Feature Selection

**Lecture reference:** `MODULE_3_OLIST_REWRITE.md` § Class 3-4

**Today:** of the 20+ features you engineered yesterday, which 8-12 actually carry signal? Drop the rest.

In [ ]:
import pandas as pd, numpy as np
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_parquet('orders_step3.parquet')
print(df.shape)

## Section 1 — Build the candidate feature matrix

In [ ]:
candidate = [
    'distance_km', 'log_freight', 'total_price', 'num_items',
    'estimate_days', 'purchase_hour', 'purchase_dayofweek',
    'is_weekend', 'days_buffer',
]
X = df[candidate].dropna()
y = df.loc[X.index, 'is_late']
print(X.shape, y.shape, '   is_late rate:', y.mean().round(3))

## Section 2 — Filter step 1: variance threshold

In [ ]:
sel = VarianceThreshold(threshold=0.01)
sel.fit(X)
kept = X.columns[sel.get_support()]
print('Kept after variance filter:', list(kept))

## Section 3 — Filter step 2: correlation matrix

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
sns.heatmap(X.corr().abs(), annot=True, fmt='.2f', cmap='coolwarm', vmin=0, vmax=1)
plt.title('Feature correlation (|r|)')
plt.show()

In [ ]:
# Drop one of any pair with |r| > 0.9
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.9)]
print('To drop (high correlation):', to_drop)
X = X.drop(columns=to_drop)
print('Remaining:', list(X.columns))

## Section 4 — Mutual information vs target

In [ ]:
mi = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print(mi_series.round(4))

## Section 5 — Embedded selection: Random Forest importances

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=12,
                            class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns) \
                  .sort_values(ascending=False)
print(importances.round(3))

## Section 6 — Final selected feature set

Cross-check the three rankings. Keep features that show up high on at least 2 of 3.

In [ ]:
FINAL = ['distance_km', 'log_freight', 'estimate_days',
        'is_weekend', 'purchase_hour', 'num_items', 'total_price']

X_final = X[FINAL]
X_final.to_parquet('X_step4_features.parquet', index=False)
y.to_frame().to_parquet('y_step4_target.parquet', index=False)
print('Saved with', X_final.shape[1], 'features')

## ✅ Quick Check

1. Two features have correlation 0.97. Why drop one?  
2. Filter selection vs embedded selection — name one advantage of each.  
3. What does PCA optimise for, and why is that not the same as 'best for prediction'?

## 📝 Homework (Bronze)

Run all cells. Submit `X_step4_features.parquet` and `y_step4_target.parquet`.

## 🔥 Homework (Gold)

Add **3 more features** (your engineered ones from M3-3 Gold). Re-run the full selection pipeline. Did your new features make it into the FINAL list? Why / why not?